The goal is to merge two datasets:
- ISIC 2019 (all records)
- ISIC 2020 (only melanoma records are extracted)

In [35]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
import shutil
from tqdm import tqdm

In [36]:
UNIFIED_DS_PATH = "/Users/mac/Development/diploma/skin-mutation-detection/unified_dataset"
UNIFIED_GT_PATH = os.path.join(UNIFIED_DS_PATH, "gt.csv")
UNIFIED_IMAGES_PATH = os.path.join(UNIFIED_DS_PATH, "images")

Go with ISIC 2019 dataset first.

In [37]:
def read_ds(gnd_path):
    df = pd.read_csv(gnd_path)
    return df

In [38]:
ISIC_2019_GROUNDTRUTH_PATH = '/Users/mac/Development/diploma/skin-mutation-detection/data/train/ISIC_2019_Training_GroundTruth.csv'
ISIC_2019_IMAGE_PATH = "/Users/mac/Development/diploma/skin-mutation-detection/data/train/ISIC_2019_Training_Input"


IMG_SIZE = (456, 456)
LABELS = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']

In [39]:
gt_2019 = read_ds(ISIC_2019_GROUNDTRUTH_PATH)

In [40]:
def move_images_to_unified_folder(source_folder, target_folder, image_names):
    """Move images from source to target folder based on list of image names."""
    os.makedirs(target_folder, exist_ok=True)
    for image_name in tqdm(image_names):
        src_path = os.path.join(source_folder, image_name + ".jpg")
        dst_path = os.path.join(target_folder, image_name + ".jpg")
        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)
        else:
            print(f"Warning: {src_path} does not exist.")

isic_2019_image_names = gt_2019['image'].tolist()

move_images_to_unified_folder(ISIC_2019_IMAGE_PATH, UNIFIED_IMAGES_PATH, isic_2019_image_names)

100%|██████████| 25331/25331 [00:35<00:00, 723.47it/s]


Then ISIC 2020

In [41]:
ISIC_2020_GROUNDTRUTH_PATH = "/Users/mac/Development/diploma/skin-mutation-detection/ISIC_2020_Dataset/ISIC_2020_Train_Metadata.csv"
ISIC_2020_IMAGE_PATH = "/Users/mac/Development/diploma/skin-mutation-detection/ISIC_2020_Dataset/train"

In [42]:
# for each record of the following format in csv
# image_name,patient_id,lesion_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target
# ISIC_2637011,IP_7279968,IL_7972535,male,45,head/neck,unknown,benign,0

# extract diagnosis and if the diagnosis is melanoma
# then convert to the following format
# image,MEL,NV,BCC,AK,BKL,DF,VASC,SCC,UNK
# ISIC_0000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0

# add all of these melanoma records to the unified groundtruth

In [50]:
isic_2020_df = pd.read_csv(ISIC_2020_GROUNDTRUTH_PATH)

melanoma_df = isic_2020_df[isic_2020_df['diagnosis'].str.lower() == 'melanoma']

def convert_to_unified_format(image_id):
    row = [0.0] * len(LABELS)
    row[LABELS.index('MEL')] = 1.0
    return [image_id] + row

melanoma_unified_records = [convert_to_unified_format(row['image_name']) for _, row in melanoma_df.iterrows()]
gt_2020 = pd.DataFrame(melanoma_unified_records, columns=['image'] + LABELS)
existing_images = set(gt_2019['image'])
gt_2020 = gt_2020[~gt_2020['image'].isin(existing_images)].reset_index(drop=True)

In [51]:
gt_2020

,image,MEL,NV,BCC,AK,BKL,DF,VASC,SCC
0,ISIC_0149568,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,ISIC_0188432,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ISIC_0207268,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ISIC_0232101,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ISIC_0247330,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
579,ISIC_9955163,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
580,ISIC_9963177,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
581,ISIC_9967383,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
582,ISIC_9978107,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [45]:
unified_groundtruth = pd.concat([gt_2019, gt_2020], ignore_index=True)

In [46]:
unified_groundtruth = unified_groundtruth.drop_duplicates(subset='image', keep='first').reset_index(drop=True)

In [47]:
gt_2019.shape, unified_groundtruth.shape

((25331, 10), (25915, 10))

In [52]:
melanoma_image_names = gt_2020['image'].tolist()

move_images_to_unified_folder(ISIC_2020_IMAGE_PATH, UNIFIED_IMAGES_PATH, melanoma_image_names)

100%|██████████| 584/584 [00:00<00:00, 588.26it/s]


In [53]:
unified_groundtruth.to_csv(UNIFIED_GT_PATH)